In [ ]:
# Standard library imports
import os
import sys
import logging
import warnings

# Third-party imports
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tools.sm_exceptions import ConvergenceWarning

# Local imports
module_path = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
if module_path not in sys.path:
    sys.path.append(module_path)
from paths import BASE_INPUT_PATH, BASE_OUTPUT_PATH


def arma_forecast(dataset, product, product_sign, price_type, output_folder):
    """
    Fits an ARMA model to forecast capacity prices for a specific product, sign, and price type.
    
    Parameters:
    -----------
    dataset : pandas.DataFrame
        The dataset containing capacity price data
    product : str
        Time product (e.g., '00_04', '04_08')
    product_sign : str
        Direction of regulation ('POS' or 'NEG')
    price_type : str
        Type of price ('AVERAGE' or 'MARGINAL')
    output_folder : pathlib.Path
        Path to save the results
        
    Returns:
    --------
    dict or None
        Dictionary with model results or None if processing failed
    """
    
    warnings.filterwarnings("ignore", category=UserWarning, 
                           message="Non-invertible starting MA parameters found.*")
    warnings.filterwarnings("ignore", category=ConvergenceWarning)
    
    # Configure logging
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )
    logger = logging.getLogger(__name__)

    logger.info(f"Processing {product_sign} {price_type} for {product}...")
     
    # Filter the dataset for the specified product
    dataset = dataset.loc[dataset['PRODUCT'] == product]
    
    # Check if filter returned any data
    if dataset.empty:
        logger.warning(f"No data found for PRODUCT={product}")
        return None
        
    # Create the column name and check if it exists
    capacity_price_column = f'{product_sign}_GERMANY_{price_type}_CAPACITY_PRICE_[(EUR/MW)/h]'
    if capacity_price_column not in dataset.columns:
        logger.warning(f"Column '{capacity_price_column}' not found in dataset")
        return None
        
    # Extract the capacity price data for the specified column
    try:
        y = dataset[capacity_price_column]
        
        # Check for NaN values
        if y.isna().any():
            logger.warning(f"NaN values found in '{capacity_price_column}', dropping them")
            y = y.dropna()
            
        # Create DataFrame with proper index
        df = pd.DataFrame({capacity_price_column: y})
        df.index = dataset.index
        
    except Exception as e:
        logger.error(f"Error processing {capacity_price_column}: {str(e)}")
        return None

    # Create a copy of the original data for reference
    df_init = df.copy()

    # Define preprocessing steps to be applied
    pre_processing = [
        '0.95 capping',
        'log transformation'
        ]

    # Apply 0.95 quantile capping to handle outliers
    if '0.95 capping' in pre_processing:
        quantile_95 = df.quantile(0.95).values[0]
        df[df > quantile_95] = quantile_95

    # Apply log transformation to make the data more normally distributed
    if 'log transformation' in pre_processing:
        # Log transformation can't handle zeros or negative values
        df = np.log(df).dropna()

    # Split the data into train, validation, and test sets
    train_limit = int(len(df) * 0.7)
    validation_limit = int(len(df) * 0.85)

    train = df.iloc[:train_limit]
    valid = df.iloc[train_limit:validation_limit]
    test = df.iloc[validation_limit:]

    # Grid search for the best ARMA parameters
    best_aic = np.inf
    best_order = None
    for p in range(5):
        for q in range(5):
            try:
                d = 0
                best_model = SARIMAX(
                    endog=train.iloc[:, 0].asfreq('D'),
                    order=(p, d, q), 
                    enforce_stationarity=False
                    )
                best_model_fit = best_model.fit()
                if best_model_fit.aic < best_aic:
                    best_aic = best_model_fit.aic
                    best_order = (p, d, q)
            except:
                continue
    
    # Train and fit the model with the best parameters
    final_model = SARIMAX(
        endog=train.iloc[:, 0].asfreq('D'),
        order=best_order,
        enforce_stationarity=False
        )
    final_model_fit = final_model.fit()

    # Forecast the validation sets
    history_valid = [x for x in train.values]
    valid_pred_d1 = []
    for i in range(len(valid)):
        model = SARIMAX(
            endog=np.array(history_valid)[:, 0],
            order=best_order,
            enforce_stationarity=False
            )
        model_fit = model.fit(disp=False)
        yhat = model_fit.forecast(
            steps=1, 
            )
        valid_pred_d1.append(yhat[0])
        history_valid.append(valid.iloc[i].values)

    # Forecast the test sets
    final_train = pd.concat([train, valid])
    history_test = [x for x in final_train.values]
    test_pred_d1 = []
    for i in range(len(test)):
        model = SARIMAX(
            endog=np.array(history_test)[:, 0],
            order=best_order,
            enforce_stationarity=False
            )
        model_fit = model.fit(disp=False)
        yhat = model_fit.forecast(
            steps=1, 
            )
        test_pred_d1.append(yhat[0])
        history_test.append(test.iloc[i].values)

    # Inverse transform the predictions
    if 'log transformation' in pre_processing:
        valid_pred_d1_inv = np.exp(valid_pred_d1)
        test_pred_d1_inv = np.exp(test_pred_d1)

    else:
        valid_pred_d1_inv = valid_pred_d1
        test_pred_d1_inv = test_pred_d1

    # Make the raw results file
    valid_init = df_init.iloc[train_limit:validation_limit]
    test_init = df_init.iloc[validation_limit:]
    valid_actual = valid_init.values
    test_actual = test_init.values

    valid_results_df = pd.DataFrame({
        'DATE': valid_init.index,
        'ACTUAL_VALUE': valid_actual[:, 0],
        'D+1': valid_pred_d1_inv,
        })

    test_results_df = pd.DataFrame({
        'DATE': test_init.index,
        'ACTUAL_VALUE': test_actual[:, 0],
        'D+1': test_pred_d1_inv
        })
    
    results_df = pd.concat([valid_results_df, test_results_df], ignore_index=True)
    results_df.to_csv(output_folder / f'arma_model_results_{product}_{product_sign}_{price_type}.csv', index=False)
    
    # Make the overview file
    output_path = output_folder / 'arma_models_overview.csv'
    if os.path.exists(output_path):
        arma_models_overview = pd.read_csv(output_path, header=0)
    else:
        arma_models_overview = pd.DataFrame()
    
    new_row = {
        'PRODUCT': product,
        'PRODUCT_SIGN': product_sign,
        'PRICE_TYPE': price_type,
        'ORDER': best_order,
        'AIC': best_aic
    }
    
    arma_models_overview = pd.concat([arma_models_overview, pd.DataFrame([new_row])], ignore_index=True)
    arma_models_overview = arma_models_overview.sort_values(by=['PRODUCT', 'PRODUCT_SIGN', 'PRICE_TYPE'])
    arma_models_overview.to_csv(output_path, index=False)

    return logger.info(f"Processing {product_sign} {price_type} for {product} completed successfully")


# Define the products, signs, and price types to process
products = ['00_04', '04_08', '08_12', '12_16', '16_20', '20_24']
product_signs = ['NEG', 'POS']
price_types = ['AVERAGE', 'MARGINAL']

# Create output folder if it doesn't exist
output_folder = BASE_OUTPUT_PATH / 'arma_model_output'
output_folder.mkdir(parents=True, exist_ok=True)

# Check if dataset exists before reading
dataset_path = BASE_INPUT_PATH / 'processed_afrr_data.csv'
if not dataset_path.exists():
    print(f"Dataset not found at {dataset_path}")
    exit(1)

# Read the dataset
try:
    processed_afrr_data = pd.read_csv(dataset_path, parse_dates=['DATE'], index_col=['DATE'])
except Exception as e:
    print(f"Error reading dataset: {e}")
    exit(1)

# Run the ARMA model for all combinations of products, signs, and price types
model_results = Parallel(n_jobs=-1)(delayed(arma_forecast)(processed_afrr_data, product, product_sign, price_type, output_folder) 
                                    for product in products 
                                    for product_sign in product_signs 
                                    for price_type in price_types)